In [ ]:
import time
import cv2
import numpy as np
from skimage.util import img_as_float
from skimage.util import img_as_ubyte


def show_in_moved_window(win_name, img, x, y):
    """
    Show an image in a window, where the position of the window can be given
    """
    cv2.namedWindow(win_name)
    cv2.moveWindow(win_name, x, y)
    cv2.imshow(win_name,img)


def capture_from_camera_and_show_images():
    print("Starting image capture")

    print("Opening connection to camera")
    url = 0
    use_droid_cam = False
    if use_droid_cam:
        url = "http://192.168.1.120:4747/video"
    cap = cv2.VideoCapture(url)
    # cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Cannot open camera")
        exit()

    print("Starting camera loop")
    # Get first image
    ret, frame = cap.read()
    # if frame is read correctly ret is True
    if not ret:
        print("Can't receive frame")
        exit()

    # Transform image to gray scale and then to float, so we can do some processing
    frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    frame_gray = img_as_float(frame_gray)

    # To keep track of frames per second
    start_time = time.time()
    n_frames = 0
    stop = False
    while not stop:
        ret, new_frame = cap.read()
        if not ret:
            print("Can't receive frame. Exiting ...")
            break

        # Transform image to gray scale and then to float, so we can do some processing
        new_frame_gray = cv2.cvtColor(new_frame, cv2.COLOR_BGR2GRAY)
        new_frame_gray = img_as_float(new_frame_gray)

        # Compute difference image
        dif_img = np.abs(new_frame_gray - frame_gray)

        # Keep track of frames-per-second (FPS)
        n_frames = n_frames + 1
        elapsed_time = time.time() - start_time
        fps = int(n_frames / elapsed_time)

        # Put the FPS on the new_frame
        str_out = f"fps: {fps}"
        font = cv2.FONT_HERSHEY_COMPLEX
        cv2.putText(new_frame, str_out, (100, 100), font, 1, 255, 1)

        # Display the resulting frame
        show_in_moved_window('Input', new_frame, 0, 10)
        show_in_moved_window('Input gray', new_frame_gray, 600, 10)
        show_in_moved_window('Difference image', dif_img, 1200, 10)

        # Old frame is updated
        frame_gray = new_frame_gray

        if cv2.waitKey(1) == ord('q'):
            stop = True

    print("Stopping image loop")
    cap.release()
    cv2.destroyAllWindows()


if __name__ == '__main__':
    capture_from_camera_and_show_images()

Starting image capture
Opening connection to camera


2025-09-26 20:23:28.013 python[344:1974588] WARNING: AVCaptureDeviceTypeExternal is deprecated for Continuity Cameras. Please use AVCaptureDeviceTypeContinuityCamera and add NSCameraUseContinuityCameraDeviceType to your Info.plist.


Starting camera loop
Stopping image loop


: 

In [2]:
# 3.
import time
import cv2
import numpy as np
from skimage.util import img_as_float
from skimage.util import img_as_ubyte

def show_in_moved_window(win_name, img, x, y):
    """
    Show an image in a window, where the position of the window can be given
    """
    cv2.namedWindow(win_name)
    cv2.moveWindow(win_name, x, y)
    cv2.imshow(win_name,img)


def capture_from_camera_and_show_images():
    print("Starting image capture")

    # Parameters
    alpha = 0.95  # Background update factor
    T = 0.1       # Threshold for binary image
    A = 0.05      # Alert threshold (5% of pixels)

    print("Opening connection to camera")
    url = 0
    use_droid_cam = False
    if use_droid_cam:
        url = "http://192.168.1.120:4747/video"
    cap = cv2.VideoCapture(url)
    if not cap.isOpened():
        print("Cannot open camera")
        exit()

    print("Starting camera loop")
    # Acquire background image
    ret, background_frame = cap.read()
    if not ret:
        print("Can't receive frame")
        exit()

    # Convert background to grayscale and float
    background_gray = cv2.cvtColor(background_frame, cv2.COLOR_BGR2GRAY)
    background_gray = img_as_float(background_gray)

    # To keep track of frames per second
    start_time = time.time()
    n_frames = 0
    stop = False
    
    while not stop:
        ret, new_frame = cap.read()
        if not ret:
            print("Can't receive frame. Exiting ...")
            break

        # Convert new image to grayscale and float
        new_frame_gray = cv2.cvtColor(new_frame, cv2.COLOR_BGR2GRAY)
        new_frame_gray = img_as_float(new_frame_gray)

        # Compute absolute difference image
        dif_img = np.abs(new_frame_gray - background_gray)

        # Create binary image by applying threshold
        binary_img = dif_img > T

        # Compute total number of foreground pixels
        F = np.sum(binary_img)
        total_pixels = binary_img.size
        
        # Compute percentage of foreground pixels
        F_percentage = F / total_pixels

        # Decide if alarm should be raised
        alarm = F_percentage > A
        
        # Create a copy of the input frame for display
        display_frame = new_frame.copy()
        
        if alarm:
            # Show alarm text on input image
            cv2.putText(display_frame, "Change Detected!", (50, 50), 
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

        # Keep track of frames-per-second (FPS)
        n_frames = n_frames + 1
        elapsed_time = time.time() - start_time
        fps = int(n_frames / elapsed_time)

        # Put the FPS and percentage on the frame
        str_out = f"fps: {fps}, F: {F_percentage*100:.2f}%"
        cv2.putText(display_frame, str_out, (100, 100), 
                   cv2.FONT_HERSHEY_COMPLEX, 0.7, (255, 255, 255), 1)

        # Convert binary image to uint8 for display
        binary_display = img_as_ubyte(binary_img)

        # Display images
        show_in_moved_window('Input', display_frame, 0, 10)
        show_in_moved_window('Background', background_gray, 300, 10)
        show_in_moved_window('Difference image', dif_img, 600, 10)
        show_in_moved_window('Binary image', binary_display, 900, 10)

        # Update background image
        background_gray = alpha * background_gray + (1 - alpha) * new_frame_gray

        if cv2.waitKey(1) == ord('q'):
            stop = True

    print("Stopping image loop")
    cap.release()
    cv2.destroyAllWindows()


if __name__ == '__main__':
    capture_from_camera_and_show_images()

Starting image capture
Opening connection to camera


2025-09-26 20:31:22.384 python[569:1982203] WARNING: AVCaptureDeviceTypeExternal is deprecated for Continuity Cameras. Please use AVCaptureDeviceTypeContinuityCamera and add NSCameraUseContinuityCameraDeviceType to your Info.plist.


Starting camera loop
Stopping image loop
Stopping image loop
